[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lorenzo-bernabe/anisotropic-heat-transfer-pinns/blob/main/notebooks/04_combined_training_minimumpointnumber.ipynb)

# Imports and Environment Setup

This section imports all required libraries for:

- Numerical computation and tensor operations
- Neural network training with PyTorch
- Sampling and data generation
- Visualization and plotting
- Experiment tracking and profiling

The computation device is automatically selected based on hardware availability:
- GPU (`cuda`) if available
- CPU otherwise

The default tensor precision is set to `float32` for efficient training and memory usage.

In [ ]:
# Visualization libraries

from pathlib import Path

import matplotlib.cm as cm
import matplotlib.pyplot as plt

# Numerical computations
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# PyTorch core modules
import torch
import torch.nn as nn

# Sampling utilities
from scipy.stats import qmc

# Learning rate scheduler
from torch.optim.lr_scheduler import StepLR

# Progress bar for training loops
from tqdm import tqdm

# Time measurement
import time

# Random number generation
import random

# Reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Use float32 precision globally for efficient GPU computation
torch.set_default_dtype(torch.float32)
dtype = torch.float32


# DEVICE CONFIGURATION
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Set the default computation device
device = get_device()
# Automatically place newly created tensors on the selected device
torch.set_default_device(device)

# Problem Definition and Physical Parameters

This section defines the physical setup of the transient heat conduction problem, including:

- Plate geometry and simulation time
- Material properties
- Heat flux boundary conditions
- Dimensionless scaling parameters
- Fiber orientations used during training

In [ ]:
# SAMPLING CONFIGURATION
# Edge samples
N = 150
# Number of collocation points
M = 750
# Number of rows of the vector for temporal weighting
ROW_TEMPORAL = 5
# Number of plotted curves in the temporal weighting
NUM_CURVES = 10

# PHYSICAL PARAMETERS
# Time [s]
Z = 100.0
# Geometry [m]
L = 0.1
# Temperature [°C]
T_IC = 20.0
T_ref = 50.0
T_max = 80.0
T_min = 20.0
delta_T = T_max - T_min
# Heat flux [W/m^2]
q_x = 0.0
q_y = 10000.0
q = torch.tensor([q_x, q_y])
# Thermal conductivity [W/(m*K)]
k_l = 40.0  # In fiber direction
k_t = 4.0  # Perpendicular to fiber
# Density [kg/m^3]
rho = 2000.0
# Heat capacity at constant pressure [J/(kg*K)]
cp = 700.0
# Reference time for nondimensionalization
Z_ref = rho * cp / k_l * L**2
# Dimensionless time
Z_dimless = Z / Z_ref

# TRAINING ANGLES
# Number of fiber orientations used during training
number_of_angles = 5
angles_to_train_example = [0, 45, 90, 135, 180]  # [°]
print(angles_to_train_example)
# Angles between 0 and 180 degrees


def wrap_angle(angle):
    return angle % 180


# Calculate dimensionless thermal diffusivity
def rotate_thermal_conductivity(k_l, k_t, theta_deg):
    theta_rad = theta_deg * torch.pi / 180.0  # Shape: [N]
    cos_theta = torch.cos(theta_rad)  # [N]
    sin_theta = torch.sin(theta_rad)  # [N]

    # Rotation matrices R: shape [N, 2, 2]
    R = torch.stack(
        [
            torch.stack([cos_theta, -sin_theta], dim=-1),
            torch.stack([sin_theta, cos_theta], dim=-1),
        ],
        dim=-2,
    )

    # K_tensor: shape [2, 2]
    K_tensor = torch.diag(torch.tensor([k_l, k_t], device=theta_deg.device))

    # Expand K_tensor to [N, 2, 2] for batch matmul
    K_tensor_batched = K_tensor.unsqueeze(0).expand(R.shape[0], -1, -1)

    # R @ K @ R^T => [N, 2, 2]
    rotated_K_tensor = R @ K_tensor_batched @ R.transpose(1, 2)

    # Inverse of each rotated K tensor
    rotated_K_tensor_inv = torch.inverse(rotated_K_tensor)

    return rotated_K_tensor, rotated_K_tensor_inv


def calculate_thermal_diffusivity_params(rotated_K_tensor, rho, cp, L, Z_ref):
    # Take first element: shape [2, 2]
    K = rotated_K_tensor[0]
    # Non dimensionalize thermal diffusivity parameters
    alpha_xx = K[0, 0] / (rho * cp) * Z_ref / L**2
    alpha_yy = K[1, 1] / (rho * cp) * Z_ref / L**2
    alpha_xy = K[0, 1] / (rho * cp) * Z_ref / L**2
    return alpha_xx, alpha_yy, alpha_xy

# Neural Network and Training Hyperparameters

This section defines the training configuration for the Physics-Informed Neural Network (PINN), including:

- Optimization parameters
- Learning rate scheduling
- Neural network architecture
- Random Fourier Feature encoding
- Adaptive loss weighting parameters

In [ ]:
# Epochs
EPOCHS = 500
# Learning rate
LR = 0.01
# Scheduler step width
STEP = 200
# Gamma factor of scheduler
GAMMA = 0.9
# Number of hidden neurons
HN = 60
# Number of hidden layers
LAYERS = 6
# Variance for Random Fourier Features
SIGMA = 1.0
# Number of Fourier Features
FEATURES = 40
# Initial weight of PDE Loss
W_PDE = 1.0
# Initial weight of Neumann loss
W_NEU = 1.0
# Initial weight of Initial Condition loss
W_IC = 1.0
# Weight update factor
ALPHA = 0.9
# Slope of temporal weights
epsilon = 1.0

# Domain Sampling Strategy (PINN Training Data)

This section defines the sampling strategy used to generate training data for the Physics-Informed Neural Network (PINN).

The dataset is composed of:

- Boundary samples (top, bottom, left, right)
- Interior collocation points (PDE residual enforcement)
- Initial condition points (t = 0)

All points are generated using Latin Hypercube Sampling (LHS) for improved space-filling properties.

Additionally, the dataset is extended to multiple fiber orientations (angles), allowing the model to learn anisotropic behavior across different material directions.

In [ ]:
def sample_domain(angle_deg):
    # Normalize the angle to [0, 1]
    theta = angle_deg / 180.0

    # Top points
    x_top = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    y_top = torch.ones((N, 1), requires_grad=True, dtype=dtype)
    t_top = Z_dimless * torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    theta_top = torch.full_like(x_top, theta, requires_grad=True, dtype=dtype)
    top = torch.column_stack([x_top, y_top, t_top, theta_top])
    top = top[top[:, 2].argsort()]  # Sort by time

    # Bottom points
    x_bottom = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    y_bottom = torch.zeros((N, 1), requires_grad=True, dtype=dtype)
    t_bottom = Z_dimless * torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    theta_bottom = torch.full_like(x_bottom, theta, requires_grad=True, dtype=dtype)
    bottom = torch.column_stack([x_bottom, y_bottom, t_bottom, theta_bottom])
    bottom = bottom[bottom[:, 2].argsort()]

    # Left points
    x_left = torch.zeros((N, 1), requires_grad=True, dtype=dtype)
    y_left = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    t_left = Z_dimless * torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    theta_left = torch.full_like(x_left, theta, requires_grad=True, dtype=dtype)
    # Exponential scaling
    y_left_exp = (torch.exp(-3 * y_left) - 1) / (
        torch.exp(torch.tensor(-3.0, dtype=dtype)) - 1
    )
    left = torch.column_stack([x_left, y_left_exp, t_left, theta_left])
    left = left[left[:, 2].argsort()]

    # Right points
    x_right = torch.ones((N, 1), requires_grad=True, dtype=dtype)
    y_right = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    t_right = Z_dimless * torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    theta_right = torch.full_like(x_right, theta, requires_grad=True, dtype=dtype)
    # Exponential scaling
    y_right_exp = (torch.exp(-3 * y_right) - 1) / (
        torch.exp(torch.tensor(-3.0, dtype=dtype)) - 1
    )
    right = torch.column_stack([x_right, y_right_exp, t_right, theta_right])
    right = right[right[:, 2].argsort()]

    # Collocation points
    points = qmc.LatinHypercube(d=3).random(M)
    # Sort the points based on the third dimension (time)
    points_sorted = points[points[:, 2].argsort()]
    x_collocation = torch.tensor(points_sorted[:, 0], requires_grad=True, dtype=dtype)
    y_collocation = torch.tensor(points_sorted[:, 1], requires_grad=True, dtype=dtype)
    t_collocation = Z_dimless * torch.tensor(
        points_sorted[:, 2], requires_grad=True, dtype=dtype
    )
    theta_collocation = torch.full_like(
        x_collocation, theta, requires_grad=True, dtype=dtype
    )
    # Exponential scaling
    y_collo_exp = (torch.exp(-3 * y_collocation) - 1) / (
        torch.exp(torch.tensor(-3.0, dtype=dtype)) - 1
    )
    collocation = torch.column_stack(
        [x_collocation, y_collo_exp, t_collocation, theta_collocation]
    )

    # Initial points
    rand_samp = qmc.LatinHypercube(d=3).random(N)
    x_t0 = torch.tensor(rand_samp[:, 0], requires_grad=True, dtype=dtype)
    y_t0 = torch.tensor(rand_samp[:, 1], requires_grad=True, dtype=dtype)
    t_t0 = torch.zeros((N, 1), requires_grad=True, dtype=dtype)
    theta_t0 = torch.full_like(x_t0, theta, requires_grad=True, dtype=dtype)
    # Exponential scaling
    y_t0_exp = (torch.exp(-3 * y_t0) - 1) / (
        torch.exp(torch.tensor(-3.0, dtype=dtype)) - 1
    )
    initial = torch.column_stack([x_t0, y_t0_exp, t_t0, theta_t0])

    return top, bottom, left, right, collocation, initial


def sample_all_domains(angles_to_train):
    top_all, bottom_all, left_all, right_all, collocation_all, initial_all = (
        [],
        [],
        [],
        [],
        [],
        [],
    )

    for angle in angles_to_train:
        top, bottom, left, right, collocation, initial = sample_domain(angle)
        top_all.append(top)
        bottom_all.append(bottom)
        left_all.append(left)
        right_all.append(right)
        collocation_all.append(collocation)
        initial_all.append(initial)

    # Concatenate data from all angles
    top = torch.cat(top_all, dim=0)
    bottom = torch.cat(bottom_all, dim=0)
    left = torch.cat(left_all, dim=0)
    right = torch.cat(right_all, dim=0)
    collocation = torch.cat(collocation_all, dim=0)
    initial = torch.cat(initial_all, dim=0)

    return top, bottom, left, right, collocation, initial


top, bottom, left, right, collocation, initial = sample_all_domains(
    angles_to_train_example
)

# 3D Visualization of Sampled Training Data

This section provides an interactive 3D visualization of all sampled training points used in the PINN.

The plot includes:
- Boundary points (top, bottom, left, right)
- Interior collocation points (PDE enforcement)
- Initial condition points (t = 0)

Each point is plotted in the normalized space:
- x, y → spatial coordinates
- t → dimensionless time
- θ → fiber orientation (encoded as color)

This visualization is used to verify:
- Proper space-time coverage
- Uniformity of sampling
- Distribution of boundary vs interior points
- Multi-angle dataset consistency

In [ ]:
def plot_interactive_3D():
    top_np = top.detach().cpu().numpy()
    bottom_np = bottom.detach().cpu().numpy()
    left_np = left.detach().cpu().numpy()
    right_np = right.detach().cpu().numpy()
    collocation_np = collocation.detach().cpu().numpy()
    initial_np = initial.detach().cpu().numpy()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=top_np[:, 0],
            y=top_np[:, 1],
            z=top_np[:, 2],
            mode="markers",
            marker=dict(
                size=4,
                color=top_np[:, 3],
                colorscale="Viridis",
                colorbar=dict(title="θ"),
            ),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=bottom_np[:, 0],
            y=bottom_np[:, 1],
            z=bottom_np[:, 2],
            mode="markers",
            marker=dict(
                size=4, color=bottom_np[:, 3], colorscale="Viridis", showscale=False
            ),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=left_np[:, 0],
            y=left_np[:, 1],
            z=left_np[:, 2],
            mode="markers",
            marker=dict(
                size=4, color=left_np[:, 3], colorscale="Viridis", showscale=False
            ),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=right_np[:, 0],
            y=right_np[:, 1],
            z=right_np[:, 2],
            mode="markers",
            marker=dict(
                size=4, color=right_np[:, 3], colorscale="Viridis", showscale=False
            ),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=collocation_np[:, 0],
            y=collocation_np[:, 1],
            z=collocation_np[:, 2],
            mode="markers",
            marker=dict(
                size=4,
                color=collocation_np[:, 3],
                colorscale="Viridis",
                showscale=False,
            ),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=initial_np[:, 0],
            y=initial_np[:, 1],
            z=initial_np[:, 2],
            mode="markers",
            marker=dict(
                size=4, color=initial_np[:, 3], colorscale="Viridis", showscale=False
            ),
        )
    )
    fig.update_layout(
        title="Interactive 3D of Sampled Points",
        scene=dict(
            xaxis_title="x [-]",
            yaxis_title="y [-]",
            zaxis_title="t[-]",
            aspectmode="cube",
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        showlegend=False,
    )

    fig.show()


plot_interactive_3D()

# Physics-Informed Neural Network (PINN) Architecture

This section defines the neural network used to approximate the temperature field.

The model is a fully connected feed-forward neural network enhanced with:

- Random Fourier Features (RFF) for improved high-frequency representation
- Hyperbolic tangent activation functions
- Multiple hidden layers for increased expressivity

The network takes as input:
- Spatial coordinates (x, y)
- Time (t)
- Fiber orientation (θ)

and outputs:
- Scalar temperature field T(x, y, t, θ)

In [ ]:
# Define the neural network


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.B = torch.normal(0.0, SIGMA, size=(4, FEATURES), device=device)
        self.layers = nn.ModuleList([nn.Linear(2 * FEATURES, HN)])

        for _ in range(LAYERS - 1):
            self.layers.append(nn.Linear(HN, HN))

        # Output layer
        self.output_layer = nn.Linear(HN, 1)

    def forward(self, x, y, t, theta_deg):
        inputs = torch.column_stack([x, y, t, theta_deg])

        pi = torch.tensor(np.pi, dtype=dtype, device=device)
        features = torch.cat(
            [torch.sin(2 * pi * inputs @ self.B), torch.cos(2 * pi * inputs @ self.B)],
            dim=-1,
        )

        for layer in self.layers:
            features = torch.tanh(layer(features))

        output = self.output_layer(features)

        return output

# Set selected seed
SEED = 0
set_seed(SEED)
net = Net()

# PDE Residual and Temporal Weighting Strategy

This section defines the Physics-Informed Neural Network (PINN) loss formulation.

It includes:

- Computation of the heat equation residual using automatic differentiation
- Handling of anisotropic thermal conductivity via rotated tensors
- Second-order spatial derivatives and time derivative
- Temporal aggregation of PDE losses
- Adaptive weighting scheme to emphasize learning across different time regimes

The goal is to enforce the governing heat conduction equation in a dimensionless anisotropic form.

In [ ]:
def pde_residual(x, y, t, phi):
    T = net(x, y, t, phi)

    T_x = torch.autograd.grad(T.sum(), x, create_graph=True, retain_graph=True)[0]
    T_xx = torch.autograd.grad(T_x.sum(), x, create_graph=True)[0]
    T_y = torch.autograd.grad(T.sum(), y, create_graph=True, retain_graph=True)[0]
    T_yy = torch.autograd.grad(T_y.sum(), y, create_graph=True)[0]
    T_xy = torch.autograd.grad(T_x.sum(), y, create_graph=True)[0]
    T_t = torch.autograd.grad(T.sum(), t, create_graph=True, retain_graph=True)[0]
    # Calculate rotated tensor and alpha components
    phi_deg = phi * 180.0
    rotated_K_tensor, rotated_K_tensor_inv = rotate_thermal_conductivity(
        k_l, k_t, phi_deg
    )
    alpha_xx, alpha_yy, alpha_xy = calculate_thermal_diffusivity_params(
        rotated_K_tensor, rho, cp, L, Z_ref
    )
    # Calculate the PDE residual
    residual_T = T_t - (alpha_xx * T_xx + alpha_yy * T_yy + 2 * alpha_xy * T_xy)
    return residual_T


def compute_pde_losses_mean(pde_losses):
    total_len = len(pde_losses)
    pde_matrix = pde_losses.reshape(
        ROW_TEMPORAL, total_len // ROW_TEMPORAL
    )  # Reshape the pde_losses to a matrix
    pde_losses_mean = torch.mean(pde_matrix, dim=1)  # Mean of the rows
    return pde_losses_mean


def compute_weighted_pde_loss(pde_losses_mean):
    # Calculate the temporal weights
    cumsum = torch.cumsum(
        pde_losses_mean, dim=0
    )  # Vector cumsum where the components are the accumulative sum of the losses
    cumsum_shifted = torch.roll(
        cumsum, shifts=1, dims=0
    )  # Shift right to get sum up to i-1
    cumsum_shifted[0] = 0.0  # Fix first value to represent empty sum
    wtemp = torch.exp(-epsilon * cumsum_shifted)

    # Calculate the weighted pde loss
    weighted_pde_losses = wtemp * pde_losses_mean
    weighted_pde_loss = torch.sum(weighted_pde_losses) / ROW_TEMPORAL
    return wtemp, weighted_pde_losses, weighted_pde_loss


wtemp_evolution = []
pde_losses_mean_evolution = []
weighted_pde_losses_evolution = []

# Loss Function: Physics-Informed Neural Network (PINN)

This section defines the full training loss used to optimize the neural network.

The total loss consists of three components:

- **Neumann boundary condition loss** (heat flux constraints)
- **Initial condition loss** (temperature at t = 0)
- **PDE residual loss** (governing heat equation enforcement)

All quantities are computed in dimensionless form.

Additionally, adaptive weighting utilities are included to stabilize training by balancing gradient contributions from different loss terms.

In [ ]:
# Define the MSE Loss function
mse = torch.nn.MSELoss()


def compute_loss(top, bottom, left, right, collocation, initial):
    # Ensure tensors are properly set up for differentiation
    x_top = top[:, 0]
    y_top = top[:, 1]
    t_top = top[:, 2]
    phi_top = top[:, 3]
    x_bottom = bottom[:, 0]
    y_bottom = bottom[:, 1]
    t_bottom = bottom[:, 2]
    phi_bottom = bottom[:, 3]
    x_left = left[:, 0]
    y_left = left[:, 1]
    t_left = left[:, 2]
    phi_left = left[:, 3]
    x_right = right[:, 0]
    y_right = right[:, 1]
    t_right = right[:, 2]
    phi_right = right[:, 3]

    # Compute predicted values
    pred_top = net(x_top, y_top, t_top, phi_top)
    pred_bottom = net(x_bottom, y_bottom, t_bottom, phi_bottom)
    pred_left = net(x_left, y_left, t_left, phi_left)
    pred_right = net(x_right, y_right, t_right, phi_right)

    # Compute the derivatives
    dpred_dx_top = torch.autograd.grad(
        pred_top.sum(), x_top, create_graph=True, retain_graph=True
    )[0]
    dpred_dy_top = torch.autograd.grad(
        pred_top.sum(), y_top, create_graph=True, retain_graph=True
    )[0]
    dpred_dx_bottom = torch.autograd.grad(
        pred_bottom.sum(), x_bottom, create_graph=True, retain_graph=True
    )[0]
    dpred_dy_bottom = torch.autograd.grad(
        pred_bottom.sum(), y_bottom, create_graph=True, retain_graph=True
    )[0]
    dpred_dx_left = torch.autograd.grad(
        pred_left.sum(), x_left, create_graph=True, retain_graph=True
    )[0]
    dpred_dy_left = torch.autograd.grad(
        pred_left.sum(), y_left, create_graph=True, retain_graph=True
    )[0]
    dpred_dx_right = torch.autograd.grad(
        pred_right.sum(), x_right, create_graph=True, retain_graph=True
    )[0]
    dpred_dy_right = torch.autograd.grad(
        pred_right.sum(), y_right, create_graph=True, retain_graph=True
    )[0]

    # Neumann boundary conditions.
    # A second-order PDE admits one scalar condition per boundary point, so the
    # *normal component* of the heat flux is prescribed rather than the full
    # flux vector: n . (alpha grad U) = q_hat on the heated top edge and 0 on
    # the insulated edges, with alpha = K / k_l the dimensionless conductivity
    # tensor. This matches the "Heat flux" and "Thermal insulation" conditions
    # used for the COMSOL reference solutions. Note the anisotropic coupling:
    # when alpha_xy != 0 an insulated edge is *not* dU/dn = 0.
    alpha_top = rotate_thermal_conductivity(k_l, k_t, phi_top * 180.0)[0] / k_l
    alpha_bottom = rotate_thermal_conductivity(k_l, k_t, phi_bottom * 180.0)[0] / k_l
    alpha_left = rotate_thermal_conductivity(k_l, k_t, phi_left * 180.0)[0] / k_l
    alpha_right = rotate_thermal_conductivity(k_l, k_t, phi_right * 180.0)[0] / k_l
    q_hat = q_y * L / (k_l * delta_T)  # Dimensionless inward flux (= 1.0 here)

    # y-component of the flux on the horizontal edges, x-component on the vertical
    res_top = (
        alpha_top[:, 1, 0] * dpred_dx_top + alpha_top[:, 1, 1] * dpred_dy_top - q_hat
    )
    res_bottom = (
        alpha_bottom[:, 1, 0] * dpred_dx_bottom
        + alpha_bottom[:, 1, 1] * dpred_dy_bottom
    )
    res_left = alpha_left[:, 0, 0] * dpred_dx_left + alpha_left[:, 0, 1] * dpred_dy_left
    res_right = (
        alpha_right[:, 0, 0] * dpred_dx_right + alpha_right[:, 0, 1] * dpred_dy_right
    )

    # Compute the individual boundary losses
    loss_top = (res_top**2).mean()
    loss_bottom = (res_bottom**2).mean()
    loss_left = (res_left**2).mean()
    loss_right = (res_right**2).mean()
    neumann_losses = loss_top + loss_bottom + loss_left + loss_right

    # Initial condition
    pred_ic = net(initial[:, 0], initial[:, 1], initial[:, 2], initial[:, 3])
    T_ic = T_IC * torch.ones_like(initial[:, 0]).unsqueeze(1)
    U_ic = (T_ic - T_ref) / delta_T
    initial_losses = mse(pred_ic, U_ic)

    # PDE residual loss
    # Loss for initial and collocation points
    initial_collo = torch.cat([initial, collocation], dim=0)
    pred_pde = pde_residual(
        initial_collo[:, 0],
        initial_collo[:, 1],
        initial_collo[:, 2],
        initial_collo[:, 3],
    )
    pde_losses = pred_pde**2

    return initial_losses, neumann_losses, pde_losses


w_pde_history = []
w_neu_history = []
w_ic_history = []


def compute_gradient_norm(loss):
    grads = torch.autograd.grad(loss, tuple(net.parameters()), allow_unused=True)
    return sum(0 if grad is None else torch.linalg.norm(grad) for grad in grads)


def update_weight(weight, grad_sum, norm):
    new_weight = grad_sum / norm
    return ALPHA * weight + (1 - ALPHA) * new_weight

# Training Loop (PINN Optimization)

This section implements the full training loop for the Physics-Informed Neural Network.

Key components:

- Adam optimizer with learning rate scheduling
- Dynamic sampling of fiber orientations per epoch
- Multi-angle training strategy
- Combined loss from:
  - Initial condition loss
  - Neumann boundary condition loss
  - Weighted PDE residual loss
- Adaptive loss rebalancing based on gradient magnitudes

The training procedure alternates between:
1. Sampling new physics-informed data
2. Computing losses across multiple fiber orientations
3. Updating network parameters
4. Periodically rebalancing loss weights

In [ ]:
# Train PINN
history = []
optimizer = torch.optim.Adam(net.parameters(), lr=LR)
scheduler = StepLR(optimizer, step_size=STEP, gamma=GAMMA)


def process_angles(angles_to_train):
    initial_l_total = torch.tensor(0.0)
    neumann_l_total = torch.tensor(0.0)
    pde_loss_weighted_total = torch.tensor(0.0)

    for angle in angles_to_train:
        # Sample the domain for each angle
        top, bottom, left, right, collocation, initial = sample_domain(angle)

        initial_l, neumann_l, pde_l = compute_loss(
            top, bottom, left, right, collocation, initial
        )

        pde_loss_mean = compute_pde_losses_mean(pde_l)
        wtemp, weighted_pde_losses, pde_loss_weighted = compute_weighted_pde_loss(
            pde_loss_mean
        )
        wtemp_evolution.append(wtemp.detach().cpu().numpy())
        pde_losses_mean_evolution.append(pde_loss_mean.detach().cpu().numpy())
        weighted_pde_losses_evolution.append(weighted_pde_losses.detach().cpu().numpy())

        initial_l_total += initial_l
        neumann_l_total += neumann_l
        pde_loss_weighted_total += pde_loss_weighted
    return initial_l_total, neumann_l_total, pde_loss_weighted_total


print("Training...")
for epoch in tqdm(range(EPOCHS)):
    angles_to_train = [
        float(round(x))
        for x in qmc.scale(
            qmc.LatinHypercube(d=1).random(number_of_angles),
            l_bounds=[0],
            u_bounds=[180],
        )
        .flatten()
        .tolist()
    ]

    # Full-batch updates per epoch: every update uses the complete collocation set
    for _ in range(int(M / 100)):
        optimizer.zero_grad()
        initial_l_total, neumann_l_total, pde_loss_weighted_total = process_angles(
            angles_to_train
        )
        loss = (
            W_IC * initial_l_total
            + W_NEU * neumann_l_total
            + W_PDE * pde_loss_weighted_total
        )
        loss.backward(retain_graph=True)
        optimizer.step()

        w_ic_history.append(W_IC)
        w_neu_history.append(W_NEU)
        w_pde_history.append(W_PDE)

        scheduler.step()
        history.append(loss.item())

    # Rebalance weights every 20 epochs
    if epoch % 20 == 0:
        initial_l_total, neumann_l_total, pde_loss_weighted_total = process_angles(
            angles_to_train
        )

        grad_ic = compute_gradient_norm(initial_l_total)
        grad_neu = compute_gradient_norm(neumann_l_total)
        grad_pde = compute_gradient_norm(pde_loss_weighted_total)
        grad_sum = grad_ic + grad_neu + grad_pde
        W_IC = update_weight(W_IC, grad_sum, grad_ic)
        W_NEU = update_weight(W_NEU, grad_sum, grad_neu)
        W_PDE = update_weight(W_PDE, grad_sum, grad_pde)

        print(f"Epoch {epoch}: W_IC={W_IC:.5f}, W_NEU={W_NEU:.5f}, W_PDE={W_PDE:.5f}")


# Plot training loss
plt.semilogy(history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()
plt.title("Training with batched sample domains")
plt.show()

# Evolution of Adaptive Loss Weights

This section visualizes how the adaptive loss weights evolve during training:

- W_IC → Initial condition loss weight  
- W_NEU → Neumann boundary condition loss weight  
- W_PDE → PDE residual loss weight  

These weights are dynamically updated based on gradient magnitudes to balance learning between different physical constraints.

Monitoring their evolution helps verify:
- stability of the adaptive weighting scheme
- dominance of any single loss term
- convergence behavior of the training process

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 5))

# Plot evolution of W_IC W_NEU and W_PDE in the same graph
plt.plot(
    [w.cpu().item() if torch.is_tensor(w) else w for w in w_ic_history],
    label="W_IC",
    color="g",
    linestyle="--",
    marker="s",
    markersize=3,
)
plt.plot(
    [w.cpu().item() if torch.is_tensor(w) else w for w in w_neu_history],
    label="W_NEU",
    color="b",
    linestyle="--",
    marker="s",
    markersize=3,
)
plt.plot(
    [w.cpu().item() if torch.is_tensor(w) else w for w in w_pde_history],
    label="W_PDE",
    color="r",
    linestyle="--",
    marker="s",
    markersize=3,
)
plt.xlabel("Epoch")
plt.ylabel("Weight Value")
plt.legend()
plt.grid()
plt.title("Evolution of W_IC, W_NEU and W_PDE over Epochs")

plt.show()

# Training Diagnostics: Temporal Loss and Weight Evolution

This section visualizes how key training quantities evolve over time:

- **wtemp evolution** → temporal weighting vector behavior
- **mean PDE losses** → raw PDE residual distribution across time bins
- **weighted PDE losses** → final weighted PDE contribution used in training

Each plot samples evenly spaced epochs to show progression during training.

These diagnostics help to understand:
- how temporal weighting adapts
- how PDE residuals evolve across time
- whether weighting improves convergence behavior

In [ ]:
def plot_wtemp_evolution(wtemp_evolution):

    # Convert list of vectors to numpy array
    wtemp_array = np.array(wtemp_evolution)
    total_epochs = len(wtemp_array)

    # Define how many curves to plot
    num_curves = NUM_CURVES

    # Evenly spaced epoch indices (rounded to int)
    epochs_to_plot = np.linspace(0, total_epochs - 1, num=num_curves, dtype=int)
    # Skip the first epoch
    epochs_to_plot = epochs_to_plot[1:]

    # Define colormap with as many colors as curves
    colormap = cm.get_cmap("viridis", num_curves)
    colors = [colormap(i) for i in range(num_curves)]

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(
        f"Evolution of the components of wtemp ({num_curves} evenly spaced epochs)",
        fontsize=14,
        fontweight="bold",
    )

    # Plot the selected vectors
    for idx, epoch in enumerate(epochs_to_plot):
        values = wtemp_array[epoch][:ROW_TEMPORAL]
        ax.plot(
            range(ROW_TEMPORAL),
            values,
            color=colors[idx],
            linewidth=2,
            label=f"Epoch {epoch+1}",
        )

    # X-axis settings
    ax.set_xticks(range(ROW_TEMPORAL))
    ax.set_xticklabels([f"wtemp{i+1}" for i in range(ROW_TEMPORAL)])
    ax.set_xlabel("Vector components")
    ax.set_ylabel("Value of wtemp component")
    ax.grid(True)

    # Add legend
    ax.legend(title="Epochs", loc="best")
    fig.tight_layout()

    # Show the figure
    plt.show()


# Call the function
plot_wtemp_evolution(wtemp_evolution)


def plot_pdelosses_evolution(pde_losses_mean_evolution):

    # Convert list of vectors to numpy array
    pdelossesmean_array = np.array(pde_losses_mean_evolution)
    total_epochs = len(pdelossesmean_array)

    # Define how many curves to plot
    num_curves = NUM_CURVES

    # Evenly spaced epoch indices (rounded to int)
    epochs_to_plot = np.linspace(0, total_epochs - 1, num=num_curves, dtype=int)
    # Skip the first epoch
    epochs_to_plot = epochs_to_plot[1:]

    # Define colormap with as many colors as curves
    colormap = cm.get_cmap("viridis", num_curves)
    colors = [colormap(i) for i in range(num_curves)]

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(
        f"Evolution of the components of the mean pde losses ({num_curves} evenly spaced epochs)",
        fontsize=14,
        fontweight="bold",
    )

    # Plot the selected vectors
    for idx, epoch in enumerate(epochs_to_plot):
        values = pdelossesmean_array[epoch][:ROW_TEMPORAL]
        ax.plot(
            range(ROW_TEMPORAL),
            values,
            color=colors[idx],
            linewidth=2,
            label=f"Epoch {epoch+1}",
        )

    # X-axis settings
    ax.set_xticks(range(ROW_TEMPORAL))
    ax.set_xticklabels([f"pde_loss_mean{i+1}" for i in range(ROW_TEMPORAL)])
    ax.set_xlabel("Vector components")
    ax.set_ylabel("Value of the mean pde losses components")
    ax.grid(True)

    # Add legend
    ax.legend(title="Epochs", loc="best")
    fig.tight_layout()

    # Show the figure
    plt.show()


# Call the function
plot_pdelosses_evolution(pde_losses_mean_evolution)


def plot_weighted_pdelosses_evolution(weighted_pde_losses_evolution):

    # Convert list of vectors to numpy array
    weightedlossesmean_array = np.array(weighted_pde_losses_evolution)
    total_epochs = len(weightedlossesmean_array)

    # Define how many curves to plot
    num_curves = NUM_CURVES

    # Evenly spaced epoch indices (rounded to int)
    epochs_to_plot = np.linspace(0, total_epochs - 1, num=num_curves, dtype=int)
    # Skip the first epoch
    epochs_to_plot = epochs_to_plot[1:]

    # Define colormap with as many colors as curves
    colormap = cm.get_cmap("viridis", num_curves)
    colors = [colormap(i) for i in range(num_curves)]

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(
        (
            "Evolution of the components of the weighted "
            f"mean pde losses ({num_curves} evenly spaced epochs)"
        ),
        fontsize=14,
        fontweight="bold",
    )

    # Plot the selected vectors
    for idx, epoch in enumerate(epochs_to_plot):
        values = weightedlossesmean_array[epoch][:ROW_TEMPORAL]
        ax.plot(
            range(ROW_TEMPORAL),
            values,
            color=colors[idx],
            linewidth=2,
            label=f"Epoch {epoch+1}",
        )

    # X-axis settings
    ax.set_xticks(range(ROW_TEMPORAL))
    ax.set_xticklabels([f"weighted_pde_loss_mean{i+1}" for i in range(ROW_TEMPORAL)])
    ax.set_xlabel("Vector components")
    ax.set_ylabel("Value of the weighted mean pde losses components")
    ax.grid(True)

    # Add legend
    ax.legend(title="Epochs", loc="best")
    fig.tight_layout()

    # Show the figure
    plt.show()


# Call the function
plot_weighted_pdelosses_evolution(weighted_pde_losses_evolution)

# Plot Training Loss

This section visualises the convergence behaviour of the PINN during the training process.

The training loss history is plotted on a logarithmic scale to capture the reduction of the loss over several orders of magnitude. The evolution of the loss is shown with respect to the number of optimisation steps.

The plot corresponds to the training strategy using batched sample domains, where multiple fibre orientation angles are considered simultaneously during training.

In [ ]:
# Plot training loss
plt.figure(figsize=(19 / 2.54, 7 / 2.54))  # 19cm x 7cm in inches
plt.semilogy(history, color="black", linewidth=1.5)  # Black line
plt.xlabel("Step", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.title("Training with batched sample domains", fontsize=11)
plt.grid(True)
plt.tight_layout()
plt.show()

# Temperature Distribution Prediction and Inference Time Evaluation

This section evaluates the trained PINN by predicting the temperature distribution for different fibre orientation angles.

The network predictions are computed for angles ranging from -60° to 180°. Since the thermal conductivity tensor is periodic with respect to the fibre orientation, the angles are wrapped to the range [0°, 180°] before being provided as input to the network.

For each investigated angle, the temperature field is evaluated at the initial time ($t = 0$ s) and at the final simulation time ($t = Z$). The predicted normalized temperatures are converted back to physical temperatures in °C and visualised as heatmaps.

Additionally, the inference time of the PINN is measured for each angle at $t = Z$. GPU synchronisation and warm-up evaluations are performed to obtain reliable timing results. The mean inference time over all investigated fibre orientations is calculated and reported.

In [ ]:
# Compute the temperature distribution for various angles, plot the heatmaps and measure inference time for each angle.
# Angles between 0 and 180 degrees
def wrap_angle(angle):
    return angle % 180


def plot_all_temperature_heatmaps(angles):
    num_points = 100
    # Create spatial grid
    val_x, val_y = np.meshgrid(np.linspace(0, 1, num_points), np.linspace(0, 1, num_points))
    X_flat = val_x.flatten()
    Y_flat = val_y.flatten()
    X_tensor = torch.tensor(X_flat, dtype=dtype, device=device)
    Y_tensor = torch.tensor(Y_flat, dtype=dtype, device=device)
    val_X = L * val_x
    val_Y = L * val_y

    # Create figure
    fig, axes = plt.subplots(len(angles), 2, figsize=(12, 5 * len(angles)))
    fig.suptitle("Temperature Distribution PINN", fontsize=18, fontweight="bold")
    # Set network to evaluation mode
    net.eval()
    # Dictionary for inference times
    inference_times = {}
    # Time at t = Z
    t_Z = torch.full_like(X_tensor, Z / Z_ref)
    # Warm-up runs
    phi_warmup = torch.zeros_like(X_tensor)

    with torch.no_grad():
        for _ in range(10):
            _ = net(
                X_tensor,
                Y_tensor,
                t_Z,
                phi_warmup
            )
    # Loop over angles

    for i, phi in enumerate(angles):
        phi_og = phi
        # Ensure phi is in [0, 180]
        phi = wrap_angle(phi_og)
        # Normalize angle
        phi_normalized = phi / 180.0
        # Create angle tensor
        phi_tensor = torch.full_like(X_tensor, phi_normalized)
        # INFERENCE TIME FOR t = Z
        # Synchronize GPU before timing
        if device.type == "cuda":
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        # PINN inference at t = Z
        with torch.no_grad():
            U_tZ = net(X_tensor, Y_tensor, t_Z, phi_tensor)
        # Synchronize GPU after timing
        if device.type == "cuda":
            torch.cuda.synchronize()
        end_time = time.perf_counter()
        # Inference time
        inference_time = end_time - start_time
        # Store inference time
        inference_times[phi_og] = inference_time
        # Print inference time
        print(f"φ = {phi_og:6.1f}° → "
            f"Inference time = "
            f"{inference_time * 1000:.3f} ms")
        # Move prediction to CPU after timing
        U_tZ = (U_tZ.detach().cpu().numpy().reshape(num_points, num_points))
        # Convert normalized temperature to actual temperature
        T_tZ = U_tZ * delta_T + T_ref
        
        # Calculate t = 0 for plotting
        t_0 = torch.zeros_like(X_tensor)
        with torch.no_grad():
            U_t0 = net(X_tensor, Y_tensor, t_0, phi_tensor)
        U_t0 = ( U_t0.detach().cpu().numpy().reshape(num_points, num_points))

        # Convert normalized temperature to actual temperature
        T_t0 = U_t0 * delta_T + T_ref

        # Plot results
        if len(angles) > 1:
            ax0 = axes[i, 0]
            ax1 = axes[i, 1]
        else:
            ax0 = axes[0]
            ax1 = axes[1]

        # t = 0
        im0 = ax0.contourf(val_X, val_Y, T_t0, cmap="viridis", levels=30)
        fig.colorbar(im0, ax=ax0, label="Temperature [°C]")
        ax0.set_title(f"φ = {phi_og}°, t = 0 [s]")
        ax0.set_xlabel("x [m]")
        ax0.set_ylabel("y [m]")
        ax0.set_aspect("equal")

        # t = Z
        im1 = ax1.contourf(val_X, val_Y, T_tZ, cmap="viridis", levels=30)
        fig.colorbar(im1, ax=ax1, label="Temperature [°C]")
        ax1.set_title(f"φ = {phi_og}°, t = {Z} [s]")
        ax1.set_xlabel("x [m]")
        ax1.set_ylabel("y [m]")
        ax1.set_aspect("equal")

    # Adjust layout
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])


    # Inference time summary
    print("\n" + "=" * 60)
    print("Inference Time Summary at t = Z")
    print("=" * 60)
    for phi, inference_time in inference_times.items():
        print(f"φ = {phi:6.1f}° → "
            f"{inference_time:.6f} s "
            f"({inference_time * 1000:.3f} ms)")
        
    # Mean inference time
    mean_time = np.mean(list(inference_times.values()))
    print("-" * 60)
    print(f"Mean inference time = "
        f"{mean_time:.6f} s "
        f"({mean_time * 1000:.3f} ms)")
    return inference_times

inference_times = plot_all_temperature_heatmaps(
    [
        -60.0, -45.0, -30.0, 0.0,
        30.0, 45.0, 60.0, 90.0,
        120.0, 135.0, 150.0, 180.0
    ]
)

# Loading COMSOL Reference Data

This section loads reference temperature field data generated with COMSOL simulations.

Each dataset corresponds to a different fiber orientation angle (-60° to 90° range).

These results are later used to validate the PINN predictions against numerical ground truth.

In [ ]:
# Path to project root: anisotropic-heat-transfer-pinns
project_root = Path.cwd().parent

# Path to comsol_data folder
comsol_data_path = project_root / "comsol_data"


comsoldataminus60 = pd.read_csv(
    comsol_data_path / "cfrpplate-60deg01m20c10000w100s.csv"
)
comsoldataminus45 = pd.read_csv(
    comsol_data_path / "cfrpplate-45deg01m20c10000w100s.csv"
)
comsoldataminus30 = pd.read_csv(
    comsol_data_path / "cfrpplate-30deg01m20c10000w100s.csv"
)
comsoldata0 = pd.read_csv(comsol_data_path / "cfrpplate0deg01m20c10000w100s.csv")
comsoldata30 = pd.read_csv(comsol_data_path / "cfrpplate30deg01m20c10000w100s.csv")
comsoldata45 = pd.read_csv(comsol_data_path / "cfrpplate45deg01m20c10000w100s.csv")
comsoldata60 = pd.read_csv(comsol_data_path / "cfrpplate60deg01m20c10000w100s.csv")
comsoldata90 = pd.read_csv(comsol_data_path / "cfrpplate90deg01m20c10000w100s.csv")

# Relative Temperature Error Distribution

This section evaluates the prediction accuracy of the PINN by comparing the predicted temperature fields with the reference COMSOL simulation data.

For each investigated fibre orientation angle, the PINN temperature predictions are calculated at the initial time ($t = 0$ s) and at the final simulation time ($t = Z$). The predicted temperatures are converted from the normalized network output to physical temperatures in °C before comparison. The relative temperature error is calculated.

The spatial distribution of the relative error is visualised using scatter plots for each fibre orientation. The error maps allow the identification of potential spatial error patterns and biases in the PINN predictions.

In [ ]:
def plot_all_error_heatmaps(angle_data_map):
    fig, axes = plt.subplots(
        len(angle_data_map), 2, figsize=(12, 5 * len(angle_data_map))
    )
    fig.suptitle(
        "Scaled Relative Temperature Error [%]", fontsize=18, fontweight="bold"
    )

    for i, (theta, comsoldata) in enumerate(angle_data_map.items()):
        X_comsol = torch.tensor(comsoldata.iloc[:, 0].values, dtype=dtype)
        Y_comsol = torch.tensor(comsoldata.iloc[:, 1].values, dtype=dtype)
        x_comsol = X_comsol / L
        y_comsol = Y_comsol / L
        theta_og = theta
        theta = wrap_angle(theta)
        theta_norm = theta / 180
        theta_tensor = torch.full_like(x_comsol, theta_norm, dtype=dtype)

        t_0 = torch.zeros_like(x_comsol)
        t_Z = torch.full_like(x_comsol, Z / Z_ref)

        T_pinn_t0 = (
            net(x_comsol, y_comsol, t_0, theta_tensor).detach().cpu() * delta_T + T_ref
        )
        T_pinn_tZ = (
            net(x_comsol, y_comsol, t_Z, theta_tensor).detach().cpu() * delta_T + T_ref
        )

        T_comsol_t0 = torch.tensor(comsoldata.iloc[:, 2].values).unsqueeze(1).cpu()
        T_comsol_tZ = torch.tensor(comsoldata.iloc[:, 12].values).unsqueeze(1).cpu()

        error_t0 = ((T_pinn_t0 - T_comsol_t0) / T_comsol_t0 * 100).squeeze()
        error_tZ = ((T_pinn_tZ - T_comsol_tZ) / T_comsol_tZ * 100).squeeze()

        ax0 = axes[i, 0] if len(angle_data_map) > 1 else axes[0]
        ax1 = axes[i, 1] if len(angle_data_map) > 1 else axes[1]

        sc0 = ax0.scatter(
            X_comsol.cpu().numpy(),
            Y_comsol.cpu().numpy(),
            c=error_t0.cpu().numpy(),
            cmap="coolwarm",
            marker="o",
        )
        fig.colorbar(sc0, ax=ax0, label="Error [%]")
        ax0.set_title(f"θ = {theta_og}°, t=0 [s]")
        ax0.set_xlabel("x [m]")
        ax0.set_ylabel("y [m]")

        sc1 = ax1.scatter(
            X_comsol.cpu().numpy(),
            Y_comsol.cpu().numpy(),
            c=error_tZ.cpu().numpy(),
            cmap="coolwarm",
            marker="o",
        )
        fig.colorbar(sc1, ax=ax1, label="Error [%]")
        ax1.set_title(f"θ = {theta_og}°, t={Z} [s]")
        ax1.set_xlabel("x [m]")
        ax1.set_ylabel("y [m]")

    plt.tight_layout(rect=(0, 0.03, 1, 0.95))


angle_data_map = {
    -60.0: comsoldataminus60,
    -45.0: comsoldataminus45,
    -30.0: comsoldataminus30,
    0.0: comsoldata0,
    30.0: comsoldata30,
    45.0: comsoldata45,
    60.0: comsoldata60,
    90.0: comsoldata90,
    120.0: comsoldataminus60,
    135.0: comsoldataminus45,
    150.0: comsoldataminus30,
    180.0: comsoldata0,
}
plot_all_error_heatmaps(angle_data_map)

# Relative Temperature Error Distribution with Global Scaling

This section evaluates the spatial distribution of the PINN prediction errors at the final simulation time ($t = Z$) and compares them against the COMSOL reference data using a common colour scale.

Before generating the error maps, a first evaluation pass is performed over all investigated fibre orientations to determine a common colour scale. The global minimum and maximum error values are calculated based on the maximum absolute error across all angles. A symmetric colour range is applied to all plots to enable a consistent comparison of positive and negative deviations between different fibre orientations.

The function additionally calculates the mean absolute error and maximum absolute error for each angle. The fibre orientations with the largest and smallest maximum absolute errors are identified and printed.

In the second evaluation pass, the error distributions are visualised using scatter plots for both the initial time ($t = 0$ s) and final simulation time ($t = Z$). The identical colour scale for all heatmaps allows direct comparison of spatial error patterns across different fibre orientations.

In [ ]:
def plot_all_error_heatmaps(angle_data_map):
    # First pass: determine global color scale
    all_error_tZ = []
    angle_errors = {}  # Store errors per angle

    for phi, comsoldata in angle_data_map.items():
        X_comsol = torch.tensor(
            comsoldata.iloc[:, 0].values, dtype=dtype, device=device
        )
        Y_comsol = torch.tensor(
            comsoldata.iloc[:, 1].values, dtype=dtype, device=device
        )
        x_comsol = X_comsol / L
        y_comsol = Y_comsol / L
        phi_og = phi
        phi = wrap_angle(phi_og)
        phi_norm = phi / 180.0
        phi_tensor = torch.full_like(x_comsol, phi_norm)
        t_Z = torch.full_like(x_comsol, Z / Z_ref)

        T_pinn_tZ = (
            net(x_comsol, y_comsol, t_Z, phi_tensor).detach().cpu() * delta_T + T_ref
        )
        T_comsol_tZ = torch.tensor(comsoldata.iloc[:, 12].values).unsqueeze(1).cpu()

        error_tZ = ((T_pinn_tZ - T_comsol_tZ) / T_comsol_tZ * 100).squeeze()
        angle_errors[phi_og] = error_tZ.cpu()
        all_error_tZ.append(error_tZ.cpu())

    # Combine all errors and find symmetric color limit
    all_error_tZ = torch.cat(all_error_tZ)
    vmax = all_error_tZ.abs().max().item()
    vmin = -vmax
    print(f"\nGlobal color scale limits: vmin={vmin:.2f}%, vmax={vmax:.2f}%")

    # Find angles with max and min of the maximal absolute errors
    max_angle = max(
        angle_errors.keys(), key=lambda k: angle_errors[k].abs().max().item()
    )
    max_error_value = angle_errors[max_angle].abs().max().item()

    min_angle = min(
        angle_errors.keys(), key=lambda k: angle_errors[k].abs().max().item()
    )
    min_error_value = angle_errors[min_angle].abs().max().item()

    print(f"Maximum absolute error = {max_error_value:.3f}% found at φ = {max_angle}°")
    print(
        f"Smallest maximum absolute error = {min_error_value:.3f}% found at φ = {min_angle}°\n"
    )

    # Second pass: generate plots
    fig, axes = plt.subplots(
        len(angle_data_map), 2, figsize=(12, 5 * len(angle_data_map))
    )
    fig.suptitle(
        "Scaled Relative Temperature Error [%]", fontsize=18, fontweight="bold"
    )

    for i, (phi, comsoldata) in enumerate(angle_data_map.items()):
        X_comsol = torch.tensor(
            comsoldata.iloc[:, 0].values, dtype=dtype, device=device
        )
        Y_comsol = torch.tensor(
            comsoldata.iloc[:, 1].values, dtype=dtype, device=device
        )
        x_comsol = X_comsol / L
        y_comsol = Y_comsol / L
        phi_og = phi
        phi = wrap_angle(phi_og)
        phi_norm = phi / 180.0
        phi_tensor = torch.full_like(x_comsol, phi_norm)

        t_0 = torch.zeros_like(x_comsol)
        t_Z = torch.full_like(x_comsol, Z / Z_ref)

        T_pinn_t0 = (
            net(x_comsol, y_comsol, t_0, phi_tensor).detach().cpu() * delta_T + T_ref
        )
        T_pinn_tZ = (
            net(x_comsol, y_comsol, t_Z, phi_tensor).detach().cpu() * delta_T + T_ref
        )

        T_comsol_t0 = torch.tensor(comsoldata.iloc[:, 2].values).unsqueeze(1).cpu()
        T_comsol_tZ = torch.tensor(comsoldata.iloc[:, 12].values).unsqueeze(1).cpu()

        error_t0 = ((T_pinn_t0 - T_comsol_t0) / T_comsol_t0 * 100).squeeze()
        error_tZ = ((T_pinn_tZ - T_comsol_tZ) / T_comsol_tZ * 100).squeeze()

        # Compute both mean and max absolute errors for this angle
        mean_error_tZ = error_tZ.abs().mean().item()
        max_error_tZ = error_tZ.abs().max().item()

        print(
            f"φ = {phi_og:6.1f}° → Mean |Error| = {mean_error_tZ:.3f}%,  Max |Error| = {max_error_tZ:.3f}%"
        )

        ax0 = axes[i, 0] if len(angle_data_map) > 1 else axes[0]
        ax1 = axes[i, 1] if len(angle_data_map) > 1 else axes[1]

        sc0 = ax0.scatter(
            X_comsol.cpu().numpy(),
            Y_comsol.cpu().numpy(),
            c=error_t0.cpu().numpy(),
            cmap="coolwarm",
            marker="o",
            vmin=vmin,
            vmax=vmax,
        )
        fig.colorbar(sc0, ax=ax0, label="Error [%]")
        ax0.set_title(f"φ = {phi_og}°, t=0 [s]")
        ax0.set_xlabel("x [m]")
        ax0.set_ylabel("y [m]")

        sc1 = ax1.scatter(
            X_comsol.cpu().numpy(),
            Y_comsol.cpu().numpy(),
            c=error_tZ.cpu().numpy(),
            cmap="coolwarm",
            marker="o",
            vmin=vmin,
            vmax=vmax,
        )
        fig.colorbar(sc1, ax=ax1, label="Error [%]")
        ax1.set_title(f"φ = {phi_og}°, t={Z} [s]")
        ax1.set_xlabel("x [m]")
        ax1.set_ylabel("y [m]")

    plt.tight_layout(rect=(0, 0.03, 1, 0.95))
    plt.show()


angle_data_map = {
    -60.0: comsoldataminus60,
    -45.0: comsoldataminus45,
    -30.0: comsoldataminus30,
    0.0: comsoldata0,
    30.0: comsoldata30,
    45.0: comsoldata45,
    60.0: comsoldata60,
    90.0: comsoldata90,
    120.0: comsoldataminus60,
    135.0: comsoldataminus45,
    150.0: comsoldataminus30,
    180.0: comsoldata0,
}
plot_all_error_heatmaps(angle_data_map)

# Spatial Error Correlation Analysis and Extreme Error Distribution

This section analyses the spatial characteristics of the PINN prediction errors by calculating the spatial covariance of the relative temperature error distribution.

The spatial covariance is computed by evaluating the correlation between the error at each point and the error at its closest neighbouring point. This provides an indication of whether errors are randomly distributed or exhibit spatial patterns. A positive spatial covariance indicates that neighbouring points tend to have errors with similar signs and magnitudes, suggesting spatially correlated error distributions.

For each investigated fibre orientation, the PINN temperature field is evaluated at the final simulation time ($t = Z$) and compared with the corresponding COMSOL reference data.

The fibre orientations with the highest and lowest MRE values are identified. Their corresponding error distributions are visualised using heatmaps with an identical colour scale to enable direct comparison.

Additionally, the spatial covariance values for these extreme cases are reported to assess whether the observed errors show spatial dependency.

In [ ]:
# Compute spatial covariance for error values
def compute_spatial_covariance(error_values, x_values, y_values):
    error_values = np.asarray(error_values)
    x_values = np.asarray(x_values)
    y_values = np.asarray(y_values)

    mean_error = np.mean(error_values)

    covariances = []

    for i in range(len(error_values)):

        # Calculate distances to all other points
        distance = np.sqrt((x_values - x_values[i])**2 +
            (y_values - y_values[i])**2)

        # Ignore the point itself
        distance[i] = np.inf

        # Find closest neighbour
        neighbour = np.argmin(distance)

        covariance = ((error_values[i] - mean_error) *
            (error_values[neighbour] - mean_error))
        covariances.append(covariance)
    return np.mean(covariances)


# Plot heatmaps for angles with highest and lowest mean relative error (MRE)
def plot_extreme_error_heatmaps(angle_data_map):
    # First pass: compute errors and MRE per angle
    angle_errors = {}
    mean_errors = {}
    spatial_covariances = {}


    for phi, comsoldata in angle_data_map.items():
        X_comsol = torch.tensor(comsoldata.iloc[:, 0].values, dtype=dtype, device=device)
        Y_comsol = torch.tensor(comsoldata.iloc[:, 1].values, dtype=dtype, device=device)
        x_comsol = X_comsol / L
        y_comsol = Y_comsol / L
        phi_bereich = wrap_angle(phi)
        phi_norm = phi_bereich / 180.0
        phi_tensor = torch.full_like(x_comsol, phi_norm)
        t_Z = torch.full_like(x_comsol, Z / Z_ref)
        T_pinn_tZ = (net(x_comsol, y_comsol, t_Z, phi_tensor).detach().cpu()* delta_T + T_ref)
        T_comsol_tZ = torch.tensor(comsoldata.iloc[:, 12].values).unsqueeze(1).cpu()

        # Relative error in %
        error_tZ = ((T_pinn_tZ - T_comsol_tZ)/ T_comsol_tZ* 100).squeeze()
        angle_errors[phi] = error_tZ

        # Mean relative error
        mean_errors[phi] = (error_tZ.abs().mean().item())

        # Spatial covariance calculation
        spatial_covariances[phi] = compute_spatial_covariance(error_tZ.numpy(), X_comsol.cpu().numpy(), Y_comsol.cpu().numpy())

        print(f"Angle {phi_bereich}°: "f"MRE = {mean_errors[phi]:.3f}%, "f"Spatial covariance = {spatial_covariances[phi]:.4f}%^2")

    # Identify angles with highest and lowest MRE
    max_mre_angle = max(mean_errors, key=lambda angle: mean_errors[angle])
    min_mre_angle = min(mean_errors, key=lambda angle: mean_errors[angle])

    print(f"\nAngle with highest MRE: "
        f"{max_mre_angle}° → "
        f"{mean_errors[max_mre_angle]:.3f}%")
    print(f"Spatial covariance: "
        f"{spatial_covariances[max_mre_angle]:.4f}%^2")
    print(f"\nAngle with lowest MRE: "
        f"{min_mre_angle}° → "
        f"{mean_errors[min_mre_angle]:.3f}%")
    print(f"Spatial covariance: "
        f"{spatial_covariances[min_mre_angle]:.4f}%^2")

    # Determine global color scale
    all_errors = torch.cat(
        [
            angle_errors[max_mre_angle],
            angle_errors[min_mre_angle]
        ]
    )


    vmax = all_errors.abs().max().item()
    vmin = -vmax


    print(f"\nGlobal color scale: "
        f"vmin={vmin:.2f}%, vmax={vmax:.2f}%"
    )

    # Generate heatmaps
    for phi in [min_mre_angle, max_mre_angle]:
        phi_bereich = wrap_angle(phi)
        comsoldata = angle_data_map[phi]
        X_comsol = torch.tensor(comsoldata.iloc[:,0].values)
        Y_comsol = torch.tensor(comsoldata.iloc[:,1].values)
        error_tZ = angle_errors[phi]
        x_dimless = X_comsol / L
        y_dimless = Y_comsol / L

        fig_width = 8 / 2.54
        fig_height = 6 / 2.54
        fig, ax = plt.subplots(figsize=(fig_width, fig_height), constrained_layout=True)

        sc = ax.scatter(x_dimless.cpu().numpy(), y_dimless.cpu().numpy(), c=error_tZ.cpu().numpy(), cmap="coolwarm", marker="o", vmin=vmin, vmax=vmax,)
        cbar = fig.colorbar(sc, ax=ax, label="RE [%]")
        cbar.ax.tick_params(labelsize=11)
        cbar.ax.yaxis.label.set_fontsize(11)

        ax.set_title(f"φ = {phi_bereich}°",fontsize=11)
        ax.set_xlabel("x", fontsize=11)
        ax.set_ylabel("y",fontsize=11)
        ax.set_xlim(x_dimless.min().item(), x_dimless.max().item())
        ax.set_ylim(y_dimless.min().item(), y_dimless.max().item())
        ax.set_xticks(
            [
                x_dimless.min().item(),
                (x_dimless.min().item()+x_dimless.max().item())/2,
                x_dimless.max().item(),
            ]
        )

        ax.set_yticks(
            [
                y_dimless.min().item(),
                (y_dimless.min().item()+y_dimless.max().item())/2,
                y_dimless.max().item(),
            ]
        )
        ax.tick_params(
            axis="both",
            labelsize=11
        )


        plt.show()



angle_data_map = {
    -60.0: comsoldataminus60,
    -45.0: comsoldataminus45,
    -30.0: comsoldataminus30,
    0.0: comsoldata0,
    30.0: comsoldata30,
    45.0: comsoldata45,
    60.0: comsoldata60,
    90.0: comsoldata90,
    120.0: comsoldataminus60,
    135.0: comsoldataminus45,
    150.0: comsoldataminus30,
    180.0: comsoldata0,
}


plot_extreme_error_heatmaps(angle_data_map)

In [ ]:
# Path to project root: anisotropic-heat-transfer-pinns
project_root = Path.cwd().parent
# Path to comsol_data folder
comsol_data_path = project_root / "comsol_data"
# Load COMSOL data for phi = 45°
comsoldata45 = pd.read_csv(comsol_data_path / "cfrpplate45deg01m20c10000w100s.csv")
# Extract COMSOL coordinates and temperature
X_comsol = torch.tensor(comsoldata45.iloc[:, 0].values, dtype=dtype)
Y_comsol = torch.tensor(comsoldata45.iloc[:, 1].values, dtype=dtype)
T_comsol = torch.tensor(comsoldata45.iloc[:, 12].values, dtype=dtype)
# Dimensionless coordinates
x_dimless = X_comsol / L
y_dimless = Y_comsol / L


# Create figure
fig_width = 8 / 2.54
fig_height = 6 / 2.54

fig, ax = plt.subplots(figsize=(fig_width, fig_height), constrained_layout=True)
# Plot COMSOL temperature field
sc = ax.scatter(x_dimless.cpu().numpy(), y_dimless.cpu().numpy(), c=T_comsol.cpu().numpy(), cmap="viridis", marker="o")
# Colorbar
cbar = fig.colorbar(sc, ax=ax, label="Temperature [°C]")
cbar.ax.tick_params(labelsize=11)
cbar.ax.yaxis.label.set_fontsize(11)

# Axis labels
ax.set_xlabel("x", fontsize=11)
ax.set_ylabel("y", fontsize=11)
# Axis limits
ax.set_xlim(x_dimless.min().item(), x_dimless.max().item())
ax.set_ylim(y_dimless.min().item(), y_dimless.max().item())
# Axis ticks
ax.set_xticks([x_dimless.min().item(), (x_dimless.min().item() + x_dimless.max().item()) / 2, x_dimless.max().item(),])
ax.set_yticks([y_dimless.min().item(), (y_dimless.min().item() + y_dimless.max().item()) / 2, y_dimless.max().item(),])
ax.tick_params(axis="both", labelsize=11)

plt.show()

In [ ]:
# Path to project root: anisotropic-heat-transfer-pinns
project_root = Path.cwd().parent
# Path to comsol_data folder
comsol_data_path = project_root / "comsol_data"
# Load COMSOL data: Only the x, y and t locations are used as input points
comsoldata45 = pd.read_csv(comsol_data_path / "cfrpplate45deg01m20c10000w100s.csv")
# Extract COMSOL coordinates
X_comsol = torch.tensor(comsoldata45.iloc[:, 0].values, dtype=dtype, device=device)
Y_comsol = torch.tensor(comsoldata45.iloc[:, 1].values, dtype=dtype, device=device)
# Dimensionless spatial coordinates
x_dimless = X_comsol / L
y_dimless = Y_comsol / L
# Define PINN inputs: Same x, y and t locations as the COMSOL points
# Time: t = Z for every COMSOL point
t_Z = torch.full_like(x_dimless, Z / Z_ref)
# Fibre orientation: phi = 45°
phi = 45.0
phi_bereich = wrap_angle(phi)
phi_norm = phi_bereich / 180.0
phi_tensor = torch.full_like(x_dimless, phi_norm)


# Predict temperature with the neural network
with torch.no_grad():
    T_pinn = (net(x_dimless, y_dimless, t_Z, phi_tensor).cpu()* delta_T + T_ref)
# Remove unnecessary dimension
T_pinn = T_pinn.squeeze()

# Create figure
fig_width = 8 / 2.54
fig_height = 6 / 2.54
fig, ax = plt.subplots(figsize=(fig_width, fig_height), constrained_layout=True)
# Plot PINN temperature field
sc = ax.scatter(x_dimless.cpu().numpy(), y_dimless.cpu().numpy(), c=T_pinn.numpy(), cmap="viridis", marker="o")
# Colorbar
cbar = fig.colorbar(sc, ax=ax, label="Temperature [°C]")
cbar.ax.tick_params(labelsize=11)
cbar.ax.yaxis.label.set_fontsize(11)
# Axis labels
ax.set_xlabel("x", fontsize=11)
ax.set_ylabel("y", fontsize=11)
# Axis limits
ax.set_xlim(x_dimless.min().item(), x_dimless.max().item())
ax.set_ylim(y_dimless.min().item(), y_dimless.max().item())
# Axis ticks
ax.set_xticks([x_dimless.min().item(),(x_dimless.min().item() + x_dimless.max().item()) / 2, x_dimless.max().item(),])
ax.set_yticks([y_dimless.min().item(),(y_dimless.min().item() + y_dimless.max().item()) / 2, y_dimless.max().item(),])
ax.tick_params(axis="both", labelsize=11)

plt.show()